# Build Validation/Test Splits for Balanced OMAMA Dataset

This notebook reloads the previously saved Hugging Face dataset that was created during fine-tuning, carves a portion of the validation split into a held-out test set, and saves the updated datasets back to scratch storage. Run all cells once before running the evaluation script.


In [1]:
from pathlib import Path
from datasets import load_from_disk, DatasetDict
import random

SCRATCH_ROOT = Path("/hpcstor6/scratch01/a/a.kanamarlapudi001/med-gemma")
SOURCE_DATASET = SCRATCH_ROOT / "hf_proc_messages_metadata"
UPDATED_DATASET = SCRATCH_ROOT / "hf_proc_messages_metadata_with_val"
TEST_DATASET = SCRATCH_ROOT / "hf_test_messages_metadata"

TEST_FRACTION = 0.3  # 30% of the previous validation set becomes test
SEED = 42


/home/a.kanamarlapudi001/miniconda3/envs/medgemma-bal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Adjust `TEST_FRACTION` if you prefer a different split (e.g. `0.2` for 20% test). Re-run this notebook whenever you want to regenerate the splits with a new seed.


In [2]:
data = load_from_disk(str(SOURCE_DATASET))
print(data)


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'messages'],
        num_rows: 9284
    })
    validation: Dataset({
        features: ['image', 'label', 'messages'],
        num_rows: 2322
    })
})


In [3]:
val = data["validation"]
split = val.train_test_split(test_size=TEST_FRACTION, seed=SEED, stratify_by_column="label")
new_val_ds = split["train"]
test_ds = split["test"]

print("Original validation size:", len(val))
print("New validation size:", len(new_val_ds))
print("Test size:", len(test_ds))


Original validation size: 2322
New validation size: 1625
Test size: 697


In [4]:
DatasetDict({
    "train": data["train"],
    "validation": new_val_ds,
}).save_to_disk(str(UPDATED_DATASET))

test_ds.save_to_disk(str(TEST_DATASET))
print("Saved updated datasets:")
print("  Train+Val ->", UPDATED_DATASET)
print("  Test      ->", TEST_DATASET)


 ... (more hidden) ...
 ... (more hidden) ...
 ... (more hidden) ...

Saved updated datasets:
  Train+Val -> /hpcstor6/scratch01/a/a.kanamarlapudi001/med-gemma/hf_proc_messages_metadata_with_val
  Test      -> /hpcstor6/scratch01/a/a.kanamarlapudi001/med-gemma/hf_test_messages_metadata
